In [2]:
import re
from time import time

import spacy
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

/home/kolla/anaconda3/envs/bro/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [19]:
def strip_html(textdata):
    
    soup = BeautifulSoup(textdata, "html.parser")
    return soup.get_text()


def clean_text(textdata):
    
    _only_letters_pattern = re.compile(r'[^A-Za-z0-9]+')
    _no_long_numbers_pattern = re.compile(r'\d{5,}')

    for i in range(len(textdata)):
    
        textdata[i] = strip_html(textdata[i])
        textdata[i] = textdata[i].lower()
        textdata[i] = _only_letters_pattern.sub(' ',textdata[i])
        textdata[i] = _no_long_numbers_pattern.sub('', textdata[i])
        textdata[i] = textdata[i].strip()
    
    return textdata


def lemmatize(textdata):

    nlp = spacy.load('en_core_web_sm')
    lemmatizer = nlp.get_pipe("lemmatizer")
    
    start = time()
    doc = nlp.pipe(textdata, batch_size=1000, n_process=4)
    end = time()

    print(f'lemmatization took {end-start} seconds')

    return doc

In [53]:
nlp = spacy.load("en_core_web_sm")
lemmatizer = nlp.get_pipe("lemmatizer")

directory = '/home/kolla/projects/imdb'
df_train = pd.read_csv(f'{directory}/imdb_train.csv')
#df_train = df_train.sample(frac=1).reset_index(drop=True)
df_test = pd.read_csv(f'{directory}/imdb_test.csv')
#df_test = df_test.sample(frac=1).reset_index(drop=True)
train_data = df_train.values.tolist()
test_data = df_test.values.tolist()

X_train = [x[0] for x in train_data]
Y_train = [x[1] for x in train_data]
X_test = [x[0] for x in test_data]
Y_test = [x[1] for x in test_data]

#X_train = X_train[:10]
#X_test = X_test[:10]
#Y_train = Y_train[:10]
#Y_test = Y_test[:10]

X_train_clean = clean_text(X_train)
X_test_clean = clean_text(X_test)

print(len(X_train_clean))

/home/kolla/anaconda3/envs/bro/lib/python3.8/site-packages/spacy/util.py:1707: UserWarning: [W111] Jupyter notebook detected: if using `prefer_gpu()` or `require_gpu()`, include it in the same cell right before `spacy.load()` to ensure that the model is loaded on the correct device. More information: http://spacy.io/usage/v3#jupyter-notebook-gpu
  warnings.warn(Warnings.W111)
<ipython-input-19-8d7b95d2e083>:3: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(textdata, "html.parser")


25000


In [59]:
X_train_doc = lemmatize(X_train_clean)

lemmatization took 6.198883056640625e-06 seconds


In [66]:
for doc, i in zip(X_train_doc, range(1)):
    print(doc[0].lemma_)
    break
    

recap


In [190]:
sentence = "I wouldnt say that it is a bad movie but i dont recommend it either"
sentence = "wouldnt shouldnt couldnt"

nlp = spacy.load('en_core_web_sm')
lemmatizer = nlp.get_pipe("lemmatizer")

doc = nlp(sentence)

parent_lemmas = [token.text for token in doc]
lem_text = [token.lemma_ for token in doc]
is_PART = np.array([token.pos_ == "PART" for token in doc], dtype=np.int32)
is_PART_index = np.where(is_PART == 1)[0]
is_PART_index = sorted(is_PART_index, reverse=True)

weights = np.random.rand(len(lem_text))
weights = list(weights)

print(parent_lemmas)
print(lem_text)
print()

for ind in is_PART_index:
    
    temp_w = weights.pop(ind)
    weights[ind-1] += temp_w

    temp_tok = parent_lemmas.pop(ind)
    parent_lemmas[ind-1] += temp_tok

print(parent_lemmas)
print(weights)

['would', 'nt', 'should', 'nt', 'could', 'nt']
['would', 'not', 'should', 'not', 'could', 'not']

['wouldnt', 'shouldnt', 'couldnt']
[1.0632922250167551, 1.006562002066052, 0.7608858144256206]


In [139]:
def clean_text(textdata):
    
    _only_letters_pattern = re.compile(r"[^A-Za-z0-9']+")
    _no_long_numbers_pattern = re.compile(r'\d{5,}')
    _no_multiple_quotes_pattern = re.compile(r"'+")

    for i in range(len(textdata)):
    
        textdata[i] = strip_html(textdata[i])
        textdata[i] = textdata[i].lower()
        textdata[i] = _only_letters_pattern.sub(' ',textdata[i])
        textdata[i] = _no_long_numbers_pattern.sub('', textdata[i])
        textdata[i] = _no_multiple_quotes_pattern.sub("", textdata[i])
        textdata[i] = textdata[i].strip()

    
    return textdata

text =  "I haven't seen you in a while 50 100 5000 @#$   ''' .    "

clean_text([text])

['i havent seen you in a while 50 100 5000']